# 02 - Configure ASSERT

Use the demo kernel after successful infrastructure deployment. The connection check calls ASSERT's own released provider client, not another inference SDK. It sends neutral connection prompts only, **not unreviewed policy tests**.

The approved current M2 uses three target models and retains GPT-5.6-sol for generation, adaptive testing and judging. Adding GPT-5.4 does not change those shared roles or regenerate the approved cases. Reusing a target as judge is not an independent assessment.

## Load generated settings

The deployment notebook must generate `env/.env` and `outputs/generated/deployment.json`. Missing or inconsistent settings fail immediately; do not copy credentials into this notebook or print the environment. Model aliases are labels, not provider model names.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

demo_root = Path('..').resolve()
sys.path.insert(0, str(demo_root))
from app.assert_lab import artifact_path, load_connections, smoke_connections

metadata = load_connections(root=demo_root)
display(pd.DataFrame([model.to_dict() for model in metadata.deployments]))

## Connect models

This makes one text and one constrained JSON call per active target through `azure/gpt5_series/<deployment-name>`. Generated settings explicitly select Entra and Azure CLI credentials; ASSERT's native token provider handles token refresh. No key or token is loaded from a demo file. Calls have a 4,096-token budget (including reasoning), a 60-second transport timeout, and no web search. Evaluation behavior remains an M2 acceptance check.

The three-model verification makes six logical billable requests; native retries may add calls. A skipped target is not passed. No alternate API version/provider/deployment is selected automatically. Never display raw native payloads or tokens.

In [ ]:
smoke_results = await smoke_connections(metadata)
smoke_path = artifact_path('connections.json', root=demo_root)
smoke_path.write_text(json.dumps({
    'checked_at': datetime.now(timezone.utc).isoformat(),
    'subscription_id': metadata.subscription_id,
    'resource_group': metadata.resource_group,
    'api_version': metadata.api_version,
    'models': smoke_results,
    'skipped_deployment': metadata.skipped_deployment.to_dict() if metadata.skipped_deployment is not None else None,
    'skip_reason': metadata.skip_reason,
}, indent=2) + '\n', encoding='utf-8')
display(pd.DataFrame(smoke_results))
print(f'Non-secret connection evidence: {smoke_path}')
if metadata.skipped_deployment is not None:
    print(f'Partial verification only: {metadata.skip_reason}')

## Define the policy

Edit [customer-support.yaml](../data/policies/customer-support.yaml): application context, concrete prohibited behavior, and permitted help. This synthetic assistant can explain refunds but has no refund-issuing tools. Keep an explicit policy ID/revision; content hashes detect edits even without a revision bump. Never include credentials or real customer information.

For another policy, copy that input, select its relative path in [lab.yaml](../data/lab.yaml), and generate/review a new suite. Earlier approved inputs and runs stay intact.

In [ ]:
from app.assert_lab import load_evaluation
policy, settings = load_evaluation(root=demo_root)
print(policy.render())
print(f'Editable policy: {demo_root / "data" / settings.policy_path}')

## Configure the evaluation

Edit [lab.yaml](../data/lab.yaml), not execution code. Defaults: **8 individual prompts + 4 adaptive scenarios**, at most **3 user/target exchanges** per scenario. Two handwritten prompts yield **14 cases per target**. Scenarios define goals and constraints, not fixed follow-up messages; ASSERT generates those during inference.

All targets receive the same policy system prompt. Native boolean **policy_violation** measures prohibited behavior; **overrefusal** measures unnecessary refusal of permitted help. One judgment per case covers the full transcript: an earlier applicable failure remains a failure after later compliance. Generated scenario-specific system prompts are explicitly removed during review so they cannot override the shared policy.

Generator and judge have 8,192-token budgets; tester and target have 4,096. These include reasoning. Native retries may add cost. A new default three-target comparison makes up to 66 target requests, 36 tester requests and 42 judge requests, plus generation. Adding only GPT-5.4 to the completed two-target comparison uses up to 22 target, 12 tester and 14 judge requests, without regenerating or reevaluating the earlier targets. Review budgets before proceeding. No temperature/reasoning overrides, web search, proxy or alternate provider path are introduced.

Changing tester/judge/turn settings requires a new approved suite. A target-only change can reuse frozen tests with fixed tester/judge, but select an actually deployed supported alias. Do not add an unplanned deployment or use the skipped alias as a fallback.

In [ ]:
from dataclasses import asdict
from app.assert_lab import provider_model
model_by_alias = {model.alias: model for model in metadata.deployments}
role_rows = [
    {'role': role, **model_by_alias[alias].to_dict(), 'provider_route': provider_model(model_by_alias[alias])}
    for role, alias in [('generator', settings.generation_target), ('tester', settings.tester_target), ('judge', settings.judge_target)]
]
display(pd.DataFrame(role_rows))
display(pd.DataFrame([model_by_alias[alias].to_dict() for alias in settings.comparison_targets]))
print(json.dumps(asdict(settings), indent=2))
print('Native configs and immutable suites: outputs/generated/configs and outputs/generated/suites')
print('Continue with 03_test.ipynb; reuse resources, do not rerun provisioning.')